# Workflow Data Science : Analyse et Matching B2B des Appels d'Offres Publics

Ce notebook rassemble les étapes du projet conformément aux consignes de `Devoir-projet-SD.pdf` et aux séances de cours.

- **Sections 1 à 3 (Binôme 1)** : Collecte, Prétraitement tabulaire/NLP et Exploration visuelle (EDA).
- **Sections 4 à 5 (Binôme 2)** : Modélisation Scikit-learn, Matching B2B et Conclusions.

In [ ]:
import sys
from pathlib import Path

# Ajout de la racine du projet au PYTHONPATH pour importer nos modules
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
from IPython.display import Image, display

from config.settings import CLEAN_DATA_PATH, FIGURES_DIR, RAW_DATA_PATH
from src.scraper.marchespublics import MarchesPublicsScraper
from src.preprocessing.tabular_cleaner import nettoyer_donnees_tabulaires
from src.preprocessing.text_cleaner import ajouter_colonne_texte_propre
from src.utils.validate_contract import valider_contrat_donnees
from src.eda.visualizer import (
    audit_qualite,
    detecter_outliers_iqr,
    generer_toutes_figures_eda,
)

## 1. Collecte des données

Les annonces d'appels d'offres et bons de commande sont collectées depuis `marchespublics.gov.ma` via `MarchesPublicsScraper` (Playwright + BeautifulSoup, avec pauses de courtoisie de 2 à 5 secondes et journalisation).

Pour garantir la **reproductibilité hors ligne** lors de l'exécution du notebook, le jeu de données brut est figé dans `data/raw/appels_offres_raw.csv`.

In [ ]:
# Pour lancer une nouvelle collecte en ligne, décommenter les 2 lignes suivantes :
# scraper = MarchesPublicsScraper()
# df_raw = scraper.collect(max_items=200)

# Chargement de la copie locale figée (mode hors ligne reproductible)
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Nombre d'annonces brutes chargées : {len(df_raw)}")
df_raw.head(3)

## 2. Prétraitement des données (Tabulaire & NLP)

1. **Nettoyage tabulaire** (`tabular_cleaner.py`) : calcul de l'empreinte unique MD5 `hash_ao`, déduplication, conversion des dates (`datetime64`), calcul de `delai_reponse_jours`, conversion des montants en `float64`, création de `estimation_manquante` et standardisation des villes/catégories.
2. **Nettoyage textuel NLP** (`text_cleaner.py`) : passage en minuscules, suppression des URLs, chiffres isolés et caractères spéciaux via `re`, puis filtrage des stopwords français (NLTK) et métier administratifs.
3. **Validation du contrat d'interface** avec le Binôme 2 avant export vers `data/processed/appels_offres_clean.csv`.

In [ ]:
df_tab = nettoyer_donnees_tabulaires(df_raw)
df_clean = ajouter_colonne_texte_propre(df_tab)

CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_DATA_PATH, index=False, encoding="utf-8-sig")
valider_contrat_donnees(CLEAN_DATA_PATH)

df_clean[["reference", "categorie", "ville", "delai_reponse_jours", "estimation_mad", "texte_propre"]].head(5)

## 3. Exploration des données (EDA)

### 3.1. Audit de qualité et détection des valeurs extrêmes (IQR)

Nous analysons d'abord les valeurs manquantes par colonne ainsi que les statistiques descriptives (`df.describe()`), puis la détection des outliers budgétaires par la méthode de l'écart interquartile (IQR).

In [ ]:
manquants, stats_desc = audit_qualite(df_clean)
display(manquants)
display(stats_desc)

print("Outliers IQR (estimation_mad) :", detecter_outliers_iqr(df_clean["estimation_mad"]))
print("Outliers IQR (caution_mad) :", detecter_outliers_iqr(df_clean["caution_mad"]))

### 3.2. Visualisations analytiques (Matplotlib & Seaborn)

Les 5 graphiques suivants sont générés dans `reports/figures/` et commentés ci-dessous :
1. **Distributions & Outliers** : forte asymétrie à droite des budgets (justifiant la transformation `log1p` pour la régression du Binôme 2).
2. **Répartition catégorielle & géographique** : poids respectif des *Fournitures*, *Services* et *Travaux*, principales villes et acheteurs publics.
3. **Tendances temporelles & Délais** : dynamique mensuelle des publications et distribution des délais de réponse par catégorie.
4. **Matrice de corrélation** : lien naturel entre estimation budgétaire et caution provisoire.
5. **Exploration textuelle** : top 20 des termes métier conservés après filtrage des stopwords.

In [ ]:
chemins_figures = generer_toutes_figures_eda(df_clean, output_dir=FIGURES_DIR)
for fig_path in chemins_figures:
    print(f"Affichage : {fig_path.name}")
    display(Image(filename=str(fig_path)))